# 00 — Data Cleaning

**The most underrated skill in data science.** Interviewers consistently report that candidates fail not on model selection, but on their ability to handle messy, real-world data confidently.

**Topics:** String cleaning, type coercion, duplicate detection & resolution, outlier detection & treatment, date parsing, inconsistent categoricals, structural problems (mixed types, malformed rows).

**Reference:** [pandas docs](https://pandas.pydata.org/docs/) | [numpy docs](https://numpy.org/doc/stable/)

**Dataset:** A deliberately dirty version of a hospital readmissions dataset — injected with realistic data quality problems.


In [ ]:
import pandas as pd
import numpy as np

# Build a deliberately dirty dataset — do not modify this cell
np.random.seed(42)
n = 2000

departments = ['Cardiology', 'cardiology', 'CARDIOLOGY', 'Oncology', 'oncology',
               'Neurology', 'NEUROLOGY', 'Pediatrics', 'pediatrics', 'Emergency']
admission_types = ['Elective', 'elective', 'ELECTIVE', 'Emergency', 'emergency',
                   'Urgent', 'urgent', 'N/A', 'na', 'unknown', None]
insurance = ['Medicare', 'Medicaid', 'Private', 'PRIVATE', 'private', 'Self-Pay',
             'self pay', 'Self pay', None, 'N/A']

raw_dates = pd.date_range('2019-01-01', periods=n, freq='6H')
date_formats = ['%Y-%m-%d', '%d/%m/%Y', '%m-%d-%Y', '%Y/%m/%d', '%d %b %Y']

dirty = pd.DataFrame({
    'patient_id': (
        list(range(1001, 1001 + n - 50)) +   # normal IDs
        list(range(1001, 1031)) +              # 30 exact duplicates
        ['  1500 ', '1501a', 'ID-1502', '??', '', None] * 4  # malformed IDs
    )[:n],
    'department': np.random.choice(departments, n),
    'admission_type': np.random.choice(admission_types, n),
    'insurance_type': np.random.choice(insurance, n),
    'age': (
        list(np.random.randint(18, 90, n - 30)) +
        [-5, 0, 150, 999, np.nan] * 6         # invalid ages
    )[:n],
    'length_of_stay': (
        list(np.random.exponential(5, n - 20)) +
        [-1, -99, 0, 500, np.nan] * 4         # invalid/outlier stays
    )[:n],
    'total_charges': (
        list(np.random.lognormal(9, 1.2, n - 40)) +
        [0, -500, np.nan, 9_999_999] * 10     # outliers & errors
    )[:n],
    'readmitted': np.random.choice(['Yes', 'yes', 'YES', 'No', 'no', 'NO',
                                    '1', '0', 'True', 'False', None, 'unknown'], n),
    'admission_date': [
        d.strftime(np.random.choice(date_formats)) for d in raw_dates
    ],
    'discharge_date': [
        (d + pd.Timedelta(days=int(np.random.exponential(5)))).strftime('%Y-%m-%d')
        if np.random.random() > 0.05 else None
        for d in raw_dates
    ],
    'diagnosis_code': (
        list(np.random.choice(['A01', 'B02', 'C03', 'D04', 'E05',
                               'F06', 'G07', 'H08'], n - 20)) +
        ['', '  ', 'UNKNOWN', 'N/A', None] * 4   # bad codes
    )[:n],
    'notes': [
        f'  Patient seen on {np.random.choice(["Mon","Tue","Wed"])}. ' +
        ('Follow-up required.  ' if np.random.random() > 0.5 else '  ') 
        for _ in range(n)
    ]
})

print(f"Dirty dataset shape: {dirty.shape}")
print(dirty.dtypes)
dirty.head(10)

---
## Exercise 1 — Structural Audit

**Before cleaning anything, audit what you have.** This is always step 1 in any real project.

Write `structural_audit(df)` returning a DataFrame indexed by column with:
- `dtype`: pandas dtype
- `null_count` and `null_pct` (rounded 2dp)
- `n_unique`: distinct value count
- `n_blank_strings`: count of cells that are empty string `''` or whitespace-only
- `sample_values`: first 3 non-null unique values as a string (comma-separated)
- `likely_type`: your inferred semantic type — one of `'id'`, `'categorical'`, `'numeric'`, `'date'`, `'text'`, `'binary'`

The `likely_type` heuristic:
- `n_unique / len(df) > 0.9` → `'id'`
- `n_unique <= 20` and dtype is object → `'categorical'`
- dtype is numeric → `'numeric'`
- column name contains `'date'` or `'time'` → `'date'`
- `n_unique <= 5` → `'binary'`
- else → `'text'`

In [ ]:
def structural_audit(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns audit DataFrame indexed by column name.
    Columns: dtype, null_count, null_pct, n_unique, n_blank_strings, sample_values, likely_type
    """
    # YOUR CODE HERE
    pass

audit = structural_audit(dirty)

In [ ]:
# --- ASSERTIONS ---
expected_cols = ['dtype', 'null_count', 'null_pct', 'n_unique', 'n_blank_strings', 'sample_values', 'likely_type']
assert list(audit.columns) == expected_cols, f"Columns mismatch: {list(audit.columns)}"
assert audit.index.tolist() == list(dirty.columns), "Index must be column names"
assert audit.loc['age', 'likely_type'] == 'numeric'
assert audit.loc['department', 'likely_type'] == 'categorical'
assert audit.loc['admission_date', 'likely_type'] == 'date'
assert audit['null_pct'].between(0, 100).all()
assert audit['n_blank_strings'].ge(0).all()
print("✓ Exercise 1 passed")
print(audit)

---
## Exercise 2 — String Standardization

**Problem:** `department`, `admission_type`, `insurance_type`, and `readmitted` have case inconsistencies, leading/trailing whitespace, and placeholder strings like `'N/A'`, `'na'`, `'unknown'`.

1. Write `standardize_categoricals(df, cols, null_placeholders)` that:
   - Strips leading/trailing whitespace from all string values.
   - Title-cases all values (e.g. `'CARDIOLOGY'` → `'Cardiology'`).
   - Replaces any value in `null_placeholders` (case-insensitive) with `np.nan`.
   - Returns a copy of `df` with those columns cleaned.

2. After standardizing, map `readmitted` to a proper boolean integer column: `1` for yes/true/1, `0` for no/false/0, `NaN` for anything else.

3. Assign result to `df_step2`.

In [ ]:
NULL_PLACEHOLDERS = ['n/a', 'na', 'unknown', 'none', '', 'null', '?', '??']
CAT_COLS = ['department', 'admission_type', 'insurance_type', 'readmitted']

def standardize_categoricals(df: pd.DataFrame, cols: list, null_placeholders: list) -> pd.DataFrame:
    """
    Strip, title-case, replace null placeholders with NaN.
    Returns copy of df.
    """
    # YOUR CODE HERE
    pass

def encode_readmitted(df: pd.DataFrame) -> pd.DataFrame:
    """
    Maps readmitted to 1/0/NaN based on yes/no/true/false/1/0 values.
    Returns copy of df.
    """
    # YOUR CODE HERE
    pass

df_step2 = standardize_categoricals(dirty, CAT_COLS, NULL_PLACEHOLDERS)
df_step2 = encode_readmitted(df_step2)

In [ ]:
# --- ASSERTIONS ---
assert df_step2['department'].str.strip().eq(df_step2['department']).all(), "No leading/trailing spaces"
assert df_step2['department'].dropna().str.istitle().all(), "Must be title-cased"
assert df_step2['department'].nunique() < dirty['department'].nunique(), "Standardization must reduce unique count"
assert set(df_step2['department'].dropna().unique()) == {'Cardiology', 'Oncology', 'Neurology', 'Pediatrics', 'Emergency'}

assert df_step2['readmitted'].dropna().isin([0, 1]).all(), "readmitted must be 0, 1, or NaN"
assert df_step2['readmitted'].isna().sum() > 0, "Some readmitted values should map to NaN"

assert len(df_step2) == len(dirty), "Row count must not change"
print(f"✓ Exercise 2 passed")
print(f"Department unique values: {sorted(df_step2['department'].dropna().unique())}")
print(f"Readmitted distribution: {df_step2['readmitted'].value_counts(dropna=False).to_dict()}")

---
## Exercise 3 — Patient ID Cleaning & Deduplication

**Problem:** `patient_id` has whitespace, non-numeric characters, malformed entries, and true duplicate records.

1. Write `clean_patient_id(series)` that:
   - Strips whitespace.
   - Removes non-numeric characters (keep only digits).
   - Converts to integer where possible, else `NaN`.
   - Returns a cleaned Series.

2. Apply to `df_step2`, store as column `patient_id_clean`.

3. Write `deduplicate(df, id_col, keep_strategy)` where `keep_strategy` is `'first'` or `'last'`:
   - Drops rows where `id_col` is NaN.
   - For duplicate IDs, keeps the row according to `keep_strategy`.
   - Returns `(df_clean, dedup_report)` where `dedup_report` is a dict: `original_rows`, `null_id_rows_dropped`, `duplicate_rows_dropped`, `final_rows`.

In [ ]:
def clean_patient_id(series: pd.Series) -> pd.Series:
    """
    Strip whitespace, extract digits only, convert to Int64 (nullable int).
    Non-convertible → NaN.
    """
    # YOUR CODE HERE
    pass

def deduplicate(df: pd.DataFrame, id_col: str, keep_strategy: str = 'first'):
    """
    Returns (df_clean, dedup_report dict).
    """
    # YOUR CODE HERE
    pass

df_step2['patient_id_clean'] = clean_patient_id(df_step2['patient_id'])
df_step3, dedup_report = deduplicate(df_step2, 'patient_id_clean', keep_strategy='first')

In [ ]:
# --- ASSERTIONS ---
assert df_step2['patient_id_clean'].dropna().apply(lambda x: str(x).isdigit()).all(), "Cleaned IDs must be numeric"
assert df_step2['patient_id_clean'].isna().sum() > 0, "Malformed IDs must become NaN"

assert set(dedup_report.keys()) == {'original_rows', 'null_id_rows_dropped', 'duplicate_rows_dropped', 'final_rows'}
assert dedup_report['original_rows'] == len(dirty)
assert dedup_report['final_rows'] == len(df_step3)
assert dedup_report['original_rows'] == (dedup_report['final_rows'] +
    dedup_report['null_id_rows_dropped'] + dedup_report['duplicate_rows_dropped'])
assert df_step3['patient_id_clean'].isna().sum() == 0, "No null IDs in deduplicated df"
assert df_step3['patient_id_clean'].duplicated().sum() == 0, "No duplicate IDs in final df"
print(f"✓ Exercise 3 passed")
print(dedup_report)

---
## Exercise 4 — Numeric Validation & Out-of-Range Handling

**Problem:** `age`, `length_of_stay`, and `total_charges` contain physically impossible values, negatives, and extreme outliers.

1. Write `validate_numeric_range(df, col, min_val, max_val, strategy)` that:
   - Replaces values outside `[min_val, max_val]` according to `strategy`:
     - `'nullify'`: set out-of-range values to `NaN`.
     - `'clip'`: clip to `[min_val, max_val]`.
     - `'flag'`: add a boolean column `{col}_flagged` marking them, but don't modify values.
   - Returns a copy of `df`.

2. Apply:
   - `age`: valid range [18, 110], strategy `'nullify'`.
   - `length_of_stay`: valid range [0, 365], strategy `'clip'`.
   - `total_charges`: valid range [1, 500_000], strategy `'flag'` then also `'nullify'`.

3. Assign result to `df_step4`.

In [ ]:
def validate_numeric_range(df: pd.DataFrame, col: str,
                            min_val: float, max_val: float,
                            strategy: str) -> pd.DataFrame:
    """
    Handle out-of-range numeric values.
    strategy: 'nullify' | 'clip' | 'flag'
    """
    # YOUR CODE HERE
    pass

df_step4 = df_step3.copy()
# YOUR CODE HERE: apply to age, length_of_stay, total_charges

In [ ]:
# --- ASSERTIONS ---
valid_ages = df_step4['age'].dropna()
assert valid_ages.between(18, 110).all(), "All non-null ages must be in [18, 110]"

los = df_step4['length_of_stay'].dropna()
assert los.min() >= 0 and los.max() <= 365, "LOS must be clipped to [0, 365]"

assert 'total_charges_flagged' in df_step4.columns, "Must add total_charges_flagged column"
assert df_step4['total_charges_flagged'].dtype == bool
assert df_step4['total_charges'].dropna().between(1, 500_000).all(), "Out-of-range charges must be nullified"

print(f"✓ Exercise 4 passed")
print(f"Age nulls after cleaning: {df_step4['age'].isna().sum()}")
print(f"Flagged total_charges: {df_step4['total_charges_flagged'].sum()}")

---
## Exercise 5 — Date Parsing & Temporal Validation

**Problem:** `admission_date` has 5 different date formats. `discharge_date` has nulls. Some discharges predate admissions.

1. Write `parse_mixed_dates(series)` that parses a Series of dates in mixed formats using `pd.to_datetime` with `infer_datetime_format` (or `dayfirst` handling). Return a datetime Series; unparseable → `NaT`.

2. Apply to both date columns.

3. Add column `los_days_derived`: `discharge_date - admission_date` in integer days. Negative values → `NaN`.

4. Add column `admission_year`, `admission_month`, `admission_dow` (0=Monday) from `admission_date`.

5. Write `temporal_consistency_report(df)` that returns a dict:
   - `n_unparseable_admission`: count of NaT in admission_date
   - `n_missing_discharge`: count of NaT in discharge_date
   - `n_discharge_before_admission`: count of rows where discharge < admission
   - `n_same_day`: count of rows where los_days_derived == 0

In [ ]:
def parse_mixed_dates(series: pd.Series) -> pd.Series:
    """
    Parse mixed-format date strings. Unparseable → NaT.
    """
    # YOUR CODE HERE
    pass

def temporal_consistency_report(df: pd.DataFrame) -> dict:
    """
    Returns temporal quality metrics dict.
    """
    # YOUR CODE HERE
    pass

df_step5 = df_step4.copy()
# YOUR CODE HERE: parse dates, add derived columns

In [ ]:
# --- ASSERTIONS ---
assert pd.api.types.is_datetime64_any_dtype(df_step5['admission_date']), "admission_date must be datetime"
assert pd.api.types.is_datetime64_any_dtype(df_step5['discharge_date']), "discharge_date must be datetime"
assert 'los_days_derived' in df_step5.columns
assert df_step5['los_days_derived'].dropna().ge(0).all(), "LOS days must be non-negative"
for col in ['admission_year', 'admission_month', 'admission_dow']:
    assert col in df_step5.columns, f"Missing {col}"
assert df_step5['admission_month'].dropna().between(1, 12).all()

report = temporal_consistency_report(df_step5)
assert set(report.keys()) == {'n_unparseable_admission', 'n_missing_discharge',
                               'n_discharge_before_admission', 'n_same_day'}
print("✓ Exercise 5 passed")
print(report)

---
## Exercise 6 — Outlier Detection & Treatment

**Problem:** Even after range validation, `total_charges` and `length_of_stay` may contain statistical outliers that distort models.

Implement 3 outlier detection methods:

1. `iqr_outliers(series, multiplier=1.5)`: returns boolean mask where `True` = outlier (beyond `Q1 - 1.5*IQR` or `Q3 + 1.5*IQR`).
2. `zscore_outliers(series, threshold=3.0)`: returns boolean mask where `|z-score| > threshold`.
3. `winsorize(series, lower_pct=0.01, upper_pct=0.99)`: clips values at the given percentiles. Returns cleaned Series.

Then:
4. Build `outlier_comparison`: DataFrame comparing IQR vs z-score outlier counts for `total_charges` and `length_of_stay`. Columns: `column`, `method`, `n_outliers`, `pct_outliers`.
5. Apply winsorization to both columns at 1st/99th percentile. Assign to `df_step6`.

In [ ]:
def iqr_outliers(series: pd.Series, multiplier: float = 1.5) -> pd.Series:
    """Returns boolean mask: True = outlier by IQR method."""
    # YOUR CODE HERE
    pass

def zscore_outliers(series: pd.Series, threshold: float = 3.0) -> pd.Series:
    """Returns boolean mask: True = outlier by z-score method."""
    # YOUR CODE HERE
    pass

def winsorize(series: pd.Series, lower_pct: float = 0.01, upper_pct: float = 0.99) -> pd.Series:
    """Clips series at given percentiles. Returns cleaned Series."""
    # YOUR CODE HERE
    pass

# YOUR CODE HERE: build outlier_comparison and df_step6
outlier_comparison = None
df_step6 = None

In [ ]:
# --- ASSERTIONS ---
s = df_step5['total_charges'].dropna()
iqr_mask = iqr_outliers(s)
assert iqr_mask.dtype == bool
assert 0 < iqr_mask.sum() < len(s), "Should flag some but not all as outliers"

z_mask = zscore_outliers(s)
assert z_mask.dtype == bool

ws = winsorize(s)
assert ws.min() >= s.quantile(0.01) - 1e-6
assert ws.max() <= s.quantile(0.99) + 1e-6

assert list(outlier_comparison.columns) == ['column', 'method', 'n_outliers', 'pct_outliers']
assert len(outlier_comparison) == 4  # 2 columns x 2 methods
assert set(outlier_comparison['method']) == {'IQR', 'zscore'}

print("✓ Exercise 6 passed")
print(outlier_comparison)

---
## Exercise 7 — Missing Value Strategy

**Problem:** After all cleaning steps, we still have nulls. Different columns warrant different imputation strategies.

1. Write `missing_value_plan(df)` that returns a DataFrame with one row per column containing nulls:
   - `null_count`, `null_pct`, `recommended_strategy`
   - Strategy logic: numeric → `'median'`; low-cardinality categorical (≤10 unique) → `'mode'`; high-cardinality categorical → `'missing_label'`; date → `'drop_row'`; binary → `'mode'`.

2. Implement `apply_imputation(df, plan_df)` that applies the recommended strategy for each column:
   - `'median'` → fill with column median.
   - `'mode'` → fill with column mode (first mode).
   - `'missing_label'` → fill with the string `'Unknown'`.
   - `'drop_row'` → drop rows where that column is null.

3. Assign to `df_clean`. Verify zero nulls remain (except columns marked drop_row that couldn't be dropped).

In [ ]:
def missing_value_plan(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns plan DataFrame for columns with nulls.
    Columns: null_count, null_pct, recommended_strategy
    """
    # YOUR CODE HERE
    pass

def apply_imputation(df: pd.DataFrame, plan_df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply imputation strategies from plan_df.
    Returns cleaned copy of df.
    """
    # YOUR CODE HERE
    pass

plan = missing_value_plan(df_step6)
df_clean = apply_imputation(df_step6, plan)

In [ ]:
# --- ASSERTIONS ---
assert list(plan.columns) == ['null_count', 'null_pct', 'recommended_strategy']
assert plan['null_count'].gt(0).all(), "Plan should only include columns with nulls"
assert plan['recommended_strategy'].isin(['median', 'mode', 'missing_label', 'drop_row']).all()

non_drop_cols = plan[plan['recommended_strategy'] != 'drop_row'].index.tolist()
for col in non_drop_cols:
    if col in df_clean.columns:
        assert df_clean[col].isna().sum() == 0, f"Column {col} still has nulls after imputation"

print(f"✓ Exercise 7 passed")
print(f"Rows before: {len(df_step6)} | Rows after: {len(df_clean)}")
print(f"Remaining nulls: {df_clean.isna().sum().sum()}")
print(plan)

---
## Exercise 8 — Cleaning Pipeline: Putting It All Together

**Task:** Wrap the full cleaning workflow into a single reproducible function.

Write `clean_hospital_data(df)` that applies all steps in sequence:
1. Standardize categoricals (Exercise 2)
2. Encode readmitted (Exercise 2)
3. Clean and deduplicate patient IDs (Exercise 3)
4. Validate numeric ranges (Exercise 4)
5. Parse dates and derive temporal features (Exercise 5)
6. Winsorize outliers (Exercise 6)
7. Impute remaining nulls (Exercise 7)

Return `(df_final, cleaning_report)` where `cleaning_report` is a dict summarizing:
- `rows_in`, `rows_out`, `rows_dropped`, `columns_added`, `nulls_before`, `nulls_after`

In [ ]:
def clean_hospital_data(df: pd.DataFrame):
    """
    Full cleaning pipeline.
    Returns (df_final, cleaning_report)
    """
    # YOUR CODE HERE
    pass

df_final, cleaning_report = clean_hospital_data(dirty)

In [ ]:
# --- ASSERTIONS ---
required_keys = {'rows_in', 'rows_out', 'rows_dropped', 'columns_added', 'nulls_before', 'nulls_after'}
assert set(cleaning_report.keys()) == required_keys
assert cleaning_report['rows_in'] == len(dirty)
assert cleaning_report['rows_out'] == len(df_final)
assert cleaning_report['rows_dropped'] == cleaning_report['rows_in'] - cleaning_report['rows_out']
assert cleaning_report['nulls_after'] < cleaning_report['nulls_before'], "Cleaning must reduce nulls"
assert cleaning_report['nulls_after'] == df_final.isna().sum().sum()
assert len(df_final) > 0, "Must have rows remaining after cleaning"

print("✓ Exercise 8 passed — Full cleaning pipeline complete")
print(cleaning_report)
print(f"\nFinal dataset: {df_final.shape}")
print(df_final.dtypes)